# Suite CNT — Containers

A container holds elements by index (`Schemas.OfContainer`): a property's value, read and written through
`Visitors.OfContainer`, its elements each through a `Visitors.OfAny`. Dense, its elements are a packed array; sparse,
blocks of them; with neither, it is dense for an integer-like index and a dict for any other hashable one. These cases
cover proxies, iterators, the value objects containers hold, and snapshots.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Comparison, JSON, Plain, Proxies, Schemas as S, Validators, YAML
from mbse.Schemas.Framework.Errors import DecodeError, ReadOnlyError
from support import listed, raises, ref, root_of, single

store = B.Store()
Int, Text, Float = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema(), store.Schemas.OfFloat.as_schema()  # the store's own
Phone = store.Schemas.OfObject(lambda o: o.properties([
    lambda t: t.name("number").schema([Text]),
    lambda t: t.name("label").schema([Text]),
]))
Cell = store.Schemas.OfObject(lambda o: o.name("Cell").properties([
    lambda t: t.name("row").schema([Int]),
    lambda t: t.name("col").schema([Int]),
]))
Tags = store.Schemas.OfContainer(lambda c: c.name("Tags").index([Int]).element([Text]))  # dense
Phones = store.Schemas.OfContainer(lambda c: c.index([Text]).element([Phone]))  # a dict
Grid = store.Schemas.OfContainer(lambda c: c.name("Grid").index([Cell]).element([Float]))  # dense, of two indices
Sparse = store.Schemas.OfContainer(lambda c: c.index([Int]).element([Text]).layout("sparse"))
Rows = store.Schemas.OfContainer(lambda c: c.index([Int]).element([Tags]))  # nested
Card = store.Schemas.OfObject(lambda o: (
    o
    .name("Card")
    .ref(True)
    .properties([
        lambda t: t.name("tags").schema([Tags]),
        lambda t: t.name("phones").schema([Phones]),
        lambda t: t.name("grid").schema([Grid]),
        lambda t: t.name("sparse").schema([Sparse]),
        lambda t: t.name("rows").schema([Rows]),
        lambda t: t.name("name").schema([Text]),
    ])
))


def container(read):
    """The container a property holds, as its visitor; None where it holds none (its visitor then visits nothing)."""
    found = []
    read.as_value(lambda a: a.as_container(lambda c: found.append(c) or c))
    return found[0] if found[0].as_schema() is not None else None


def scalars(c):
    """A container's elements present, each with its index, as scalars."""
    out = []
    c.as_entries(lambda pairs: out.extend((index, value_of_any(e)) for index, e in pairs))
    return out


def value_of_any(e):
    found = []
    e.as_scalar(lambda s: found.append(s.get()) or s)
    return found[0] if found else None


def walk(c):
    """Each position of a dense or sparse container, with its element's scalar, by its iterator."""
    out = []

    def visit(it):
        it.first()
        while it.index() is not None:
            values = []
            it.as_item(lambda e: e.as_scalar(lambda s: values.append(s.get()) or s))
            out.append((it.index(), values[0]))
            it.next()
    c.as_ordered(visit)
    return out

## CNT-01 · Elements by index: written through a builder's container, read through a data instance's; reads never raise

In [ ]:
card = store.Card(lambda c: (
    c
    .tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a") and t.at(2).as_scalar("c"))))
    .phones(lambda p: p.as_value(lambda a: a.as_container(lambda d: d.at("home").as_object(lambda r: r.number("1")))))
    .grid(lambda p: p.as_value(lambda a: a.as_container(lambda g: g.at((1, 2)).as_scalar(3.5))))
))
tags, phones, grid = container(card.tags), container(card.phones), container(card.grid)
assert tags.as_schema() is Tags and (len(tags), len(phones), len(grid)) == (3, 1, 6)  # dense: every index up to the last
assert scalars(tags) == [(0, "a"), (2, "c")] and value_of_any(tags.at(1)) is None  # an index with nothing reads as empty
assert walk(tags) == [(0, "a"), (1, None), (2, "c")]  # an iterator visits every position of a dense container
assert walk(grid) == [((0, 0), None), ((0, 1), None), ((0, 2), None), ((1, 0), None), ((1, 1), None), ((1, 2), 3.5)]
seen = []
phones.as_dict(lambda d: seen.append(sorted(d)))
tags.as_dict(lambda d: seen.append("not a dict"))  # not called: tags is dense
phones.as_ordered(lambda it: seen.append("not ordered"))  # not called: phones is a dict
assert seen == [["home"]]
assert str(phones.at("home").as_object(lambda r: seen.append(str(r.number)) or r) and seen[-1]) == "1"
assert container(store.Card(lambda c: c).tags) is None  # not set
card.name.as_value(lambda a: a.as_container(lambda c: seen.append((c.as_schema(), len(c), c.at(0).as_schema()))))
assert seen[-1] == (None, 0, None)  # a value of another kind: an empty visitor, read as an empty container
card.name.as_value(lambda a: a.as_container(lambda c: c.at(0).as_container(lambda x: seen.append(x.as_schema()))))
assert seen[-1] is None
empty = []
card.name.as_value(lambda a: a.as_container(lambda c: c.as_entries(lambda pairs: empty.append(list(pairs))).as_ordered(
    lambda it: empty.append("x")).as_dict(lambda d: empty.append("x"))))
assert empty == [[]]
with raises(ReadOnlyError, match="the container is read-only; write through a builder"):
    tags.remove(0)
with raises(ReadOnlyError, match="the container[0] is read-only; write through a builder"):
    tags.at(0).as_scalar("z")
with raises(ReadOnlyError, match="property 'tags' is read-only; write through a builder"):
    card.tags.as_value(lambda a: a.as_container(tags))
with raises(ReadOnlyError, match="property 'name' does not hold a container"):
    store.Card(lambda c: c.name(lambda p: p.as_value(lambda a: a.as_container(tags))))
assert repr(tags) == "<container of 2>"
# A builder edits a copy, which it reads back; an update writes it, a clone copies it.
edited = store.Card(card, lambda c: c.tags(lambda p: p.as_value(lambda a: a.as_container(
    lambda t: t.remove(2).at(1).as_scalar("b") and len(t) == 2 and t))))
assert edited is card and scalars(container(card.tags)) == [(0, "a"), (1, "b")]
copy = store.Card(card)
assert scalars(container(copy.tags)) == [(0, "a"), (1, "b")] and container(copy.tags) is not container(card.tags)
# A container given as a value is written as it is; one of another schema is refused.
given = store.Card(lambda c: c.tags(lambda p: p.as_value(lambda a: a.as_container(container(card.tags)))))
assert scalars(container(given.tags)) == [(0, "a"), (1, "b")]
with raises(TypeError, match="property 'tags' takes a container of its schema or a Spec, got _ContainerData"):
    store.Card(lambda c: c.tags(lambda p: p.as_value(lambda a: a.as_container(container(card.grid)))))
assert scalars(container(store.Card(lambda c: c.tags(container(card.tags))).tags)) == [(0, "a"), (1, "b")]  # as a property's value
twice = store.Card(lambda c: (
    c
    .tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a"))))
    .tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(1).as_scalar("b"))))
))
assert scalars(container(twice.tags)) == [(0, "a"), (1, "b")]  # a builder reads back, and adds to, what it has written
# Containers nest: a container's element may be a container.
nested = store.Card(lambda c: c.rows(lambda p: p.as_value(lambda a: a.as_container(
    lambda r: r.at(1).as_container(lambda t: t.at(0).as_scalar("x"))))))
inner = []
container(nested.rows).at(1).as_container(lambda t: inner.append(scalars(t)))
assert inner == [[(0, "x")]] and len(container(nested.rows)) == 2

## CNT-02 · Iterators: positions before the first, at each, and past the last; a builder's extends a container and inserts into one of int indices

In [ ]:
sparse = store.Card(lambda c: c.sparse(lambda p: p.as_value(lambda a: a.as_container(
    lambda c: c.at(10).as_scalar("j") and c.at(11).as_scalar("k") and c.at(40).as_scalar("z")))))
assert walk(container(sparse.sparse)) == [(10, "j"), (11, "k"), (40, "z")] and len(container(sparse.sparse)) == 3
moves = []
container(sparse.sparse).as_ordered(lambda it: moves.extend([
    it.index(), it.first().index(), it.next().index(), it.last().index(), it.next().index(), it.next().index(),
    it.prev().index(), it.first().prev().index(), it.prev().index(), repr(it)]))
assert moves == [None, 10, 11, 40, None, None, 40, None, None, "<iterator at None>"]
calls = []
container(sparse.sparse).as_ordered(lambda it: it.last().next().as_item(lambda e: calls.append(e)))
assert calls == []  # past the end of a read-only container: not called
with raises(ReadOnlyError, match="the container is read-only; write through a builder"):
    container(sparse.sparse).as_ordered(lambda it: it.first().prepend("x"))


def edit(card, name, cb):
    """Edits a card's container through its builder's iterator."""
    return store.Card(card, lambda c: getattr(c, name)(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(cb)))))


edit(sparse, "sparse", lambda it: it.last().next().as_item(lambda e: e.as_scalar("past")).first().prev().as_item(
    lambda e: e.as_scalar("before")))
assert walk(container(sparse.sparse)) == [(10, "before"), (11, "j"), (12, "k"), (41, "z"), (42, "past")]
edit(sparse, "sparse", lambda it: it.first().append("after").prepend(lambda e: e.as_scalar("in")))
assert walk(container(sparse.sparse)) == [(10, "before"), (11, "in"), (12, "after"), (13, "j"), (14, "k"), (43, "z"), (44, "past")]
fresh = store.Card(lambda c: c.tags(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(
    lambda it: it.as_item(lambda e: e.as_scalar("first")).next().as_item(lambda e: e.as_scalar("second")))))))
assert walk(container(fresh.tags)) == [(0, "first"), (1, "second")]  # an empty container extended from its ends
edit(fresh, "tags", lambda it: it.last().next().append("end").first().prev().prepend("start"))
assert walk(container(fresh.tags)) == [(0, "start"), (1, "first"), (2, "second"), (3, "end")]
blank = store.Card(lambda c: c.tags(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(
    lambda it: it.append("only"))))))
assert walk(container(blank.tags)) == [(0, "only")]
ends = store.Card(lambda c: c.tags(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(
    lambda it: it.first().as_item(lambda e: e.as_scalar("a")).last().next().prepend("z").first().prev().append("y"))))))
assert walk(container(ends.tags)) == [(0, "y"), (1, "a"), (2, "z")]  # at the ends, of an empty container and of one with elements
lone = store.Card(lambda c: c.tags(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.as_ordered(lambda it: it.prepend("p"))))))
assert walk(container(lone.tags)) == [(0, "p")]
gridded = store.Card(card, lambda c: c.grid(lambda p: p.as_value(lambda a: a.as_container(lambda g: g.as_ordered(
    lambda it: it.first().as_item(lambda e: e.as_scalar(0.5)))))))
assert walk(container(gridded.grid))[0] == ((0, 0), 0.5)
with raises(ValueError, match="property 'grid': only a dense or sparse container of int indices has positions to insert at; write through at(index)"):
    edit(card, "grid", lambda it: it.first().append(1.0))

## CNT-03 · Value objects in containers belong to the object holding the container: edited in place, copied by a clone, removed with their element

In [ ]:
owner = store.Card(lambda c: c.phones(lambda p: p.as_value(lambda a: a.as_container(
    lambda d: d.at("home").as_object(lambda r: r.number("1").label("h")) and d.at("work").as_object(lambda r: r.number("2"))))))
held = {}
container(owner.phones).as_dict(lambda d: held.update({k: e for k, e in d.items()}))
home = []
held["home"].as_object(lambda r: home.append(r) or r)
assert Proxies.owner_of(home[0]) is owner and str(home[0].label) == "h"
store.Card(owner, lambda c: c.phones(lambda p: p.as_value(lambda a: a.as_container(lambda d: d.at("home").as_object(lambda r: r.label("x"))))))
again = []
container(owner.phones).at("home").as_object(lambda r: again.append(r) or r)
assert again[0] is home[0] and str(home[0].label) == "x" and str(home[0].number) == "1"  # edited in place
twin = store.Card(owner)
copied = []
container(twin.phones).at("home").as_object(lambda r: copied.append(r) or r)
assert copied[0] is not home[0] and Proxies.owner_of(copied[0]) is twin and str(copied[0].number) == "1"
store.Card(owner, lambda c: c.phones(lambda p: p.as_value(lambda a: a.as_container(lambda d: d.remove("home")))))
assert len(container(owner.phones)) == 1 and Proxies.owner_of(home[0]) is owner
Kit = store.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("parts").schema([Tags])]))  # a value object holding a container
Box = store.Schemas.OfObject(lambda o: o.name("Box").ref(True).properties([lambda t: t.name("kit").schema([Kit])]))
box = store.Box(lambda b: b.kit(lambda p: p.as_object(lambda k: k.parts(lambda q: q.as_value(lambda a: a.as_container(
    lambda t: t.at(0).as_scalar("bolt")))))))
twin_box = store.Box(box)
parts = []
twin_box.kit.as_object(lambda k: (k.parts.as_value(lambda a: a.as_container(lambda t: parts.append(scalars(t)))), k)[1])
assert parts == [[(0, "bolt")]]  # copied with the value object that holds it
moved = store.Card(lambda c: c.tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a")))))
store.Card(moved, lambda c: c.tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.as_ordered(lambda it: it.first().prepend("z"))))))
assert walk(container(moved.tags)) == [(0, "z"), (1, "a")]

## CNT-04 · Snapshots: a dense container as an array, nested or flat for several indices, a sparse one as blocks, a dict by text as a mapping, entries otherwise; read back, and over what a store holds

In [ ]:
card = store.Card(lambda c: (
    c
    .tags(lambda p: p.as_value(lambda a: a.as_container(lambda t: t.at(0).as_scalar("a") and t.at(2).as_scalar("c"))))
    .phones(lambda p: p.as_value(lambda a: a.as_container(lambda d: d.at("home").as_object(lambda r: r.number("1")))))
    .grid(lambda p: p.as_value(lambda a: a.as_container(lambda g: g.at((1, 2)).as_scalar(3.5) and g.at((0, 0)).as_scalar(0.25))))
    .sparse(lambda p: p.as_value(lambda a: a.as_container(lambda s: s.at(3).as_scalar("x") and s.at(4).as_scalar("y") and s.at(9).as_scalar("z"))))
    .rows(lambda p: p.as_value(lambda a: a.as_container(lambda r: r.at(0).as_container(lambda t: t.at(1).as_scalar("n")))))
))
written = root_of(Plain.ToPlain(store).OfObject(Card, card))
assert len(container(card.grid)) == 6  # the box its indices span
assert written == {"tags": ["a", None, "c"], "phones": {"home": {"number": "1"}}, "grid": [[0.25, None, None], [None, None, 3.5]],
                   "sparse": [{"at": 3, "items": ["x", "y"]}, {"at": 9, "items": ["z"]}], "rows": [[None, "n"]]}
flat = root_of(Plain.ToPlain(store, flat=True).OfObject(Card, card))
assert flat["grid"] == {"shape": [2, 3], "items": [0.25, None, None, None, None, 3.5]}
assert '"grid": {"shape": [2, 3]' in JSON.ToJSON(store, flat=True).OfObject(Card, card) and "shape:" in YAML.ToYAML(store, flat=True).OfObject(Card, card)
other = B.Store()  # another store of the same schemas
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot())  # store's schemas, as they are now: other's own
for snapshot in (Plain.ToPlain(store).OfObject(Card, card), Plain.ToPlain(store, flat=True).OfObject(Card, card)):
    back = Plain.FromPlain(other).OfObject(other.Card, snapshot)  # the loader takes either form
    assert root_of(Plain.ToPlain(other).OfObject(other.Card, back)) == written
again = Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card))
assert again is not card and root_of(Plain.ToPlain(store).OfObject(Card, again)) == written  # reloading a counted object makes a new one
# Any other index is written as entry-like elements: a dict by bytes, a dense container by text, a sparse one by text.
ByBytes = store.Schemas.OfContainer(lambda c: c.index([store.Schemas.OfBytes.as_schema()]).element([Int]))
Sorted = store.Schemas.OfContainer(lambda c: c.index([Text]).element([Int]).layout("dense"))
Keyed = store.Schemas.OfContainer(lambda c: c.index([Cell]).element([Int]).layout("sparse"))
Mixed = store.Schemas.OfObject(lambda o: o.properties([
    lambda t: t.name("k").schema([Text]),
    lambda t: t.name("n").schema([Int]),
]))
ByMixed = store.Schemas.OfContainer(lambda c: c.index([Mixed]).element([Int]))
Note = store.Schemas.OfObject(lambda o: (
    o
    .name("Note")
    .ref(True)
    .properties([
        lambda t: t.name("b").schema([ByBytes]),
        lambda t: t.name("s").schema([Sorted]),
        lambda t: t.name("k").schema([Keyed]),
        lambda t: t.name("m").schema([ByMixed]),
    ])
))
note = store.Note(lambda n: (
    n
    .b(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(b"\x01").as_scalar(1))))
    .s(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at("b").as_scalar(2) and c.at("a").as_scalar(1))))
    .k(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at((0, 1)).as_scalar(1) and c.at((0, 2)).as_scalar(2) and c.at((3, 0)).as_scalar(3))))
    .m(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(("x", 1)).as_scalar(5))))
))
noted = root_of(Plain.ToPlain(store).OfObject(Note, note))
assert noted == {"b": [{"index": "AQ==", "element": 1}], "s": [{"index": "a", "element": 1}, {"index": "b", "element": 2}],
                 "k": [{"at": {"row": 0, "col": 1}, "items": [1, 2]}, {"at": {"row": 3, "col": 0}, "items": [3]}],
                 "m": [{"index": {"k": "x", "n": 1}, "element": 5}]}
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot())  # store's schemas, as they are now: other's own
assert root_of(Plain.ToPlain(other).OfObject(other.Note, Plain.FromPlain(other).OfObject(other.Note, Plain.ToPlain(store).OfObject(Note, note)))) == noted
# A union's branch, or an intersection's part, may hold a container.
Either = store.Schemas.OfUnion(lambda u: u.branches([
    lambda t: t.name("tags").schema([Tags]),
    lambda t: t.name("word").schema([Text]),
]))
Choice = store.Schemas.OfObject(lambda o: o.name("Choice").ref(True).properties([lambda t: t.name("pick").schema([Either])]))
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot())  # store's schemas, as they are now: other's own
choice = store.Choice(lambda c: c.pick(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.tags(lambda t: t.at(1).as_scalar("q"))))))
picked = root_of(Plain.ToPlain(store).OfObject(Choice, choice))
assert picked == {"pick": {"tags": [None, "q"]}}
assert root_of(Plain.ToPlain(other).OfObject(other.Choice, Plain.FromPlain(other).OfObject(other.Choice, Plain.ToPlain(store).OfObject(Choice, choice)))) == picked
store.Choice(choice, lambda c: c.pick(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.tags(lambda t: t.at(2).as_scalar("r"))))))
assert root_of(Plain.ToPlain(store).OfObject(Choice, choice)) == {"pick": {"tags": [None, "q", "r"]}}  # its container edited
same_branch = store.Choice(lambda c: c.pick(lambda p: p.as_value(lambda a: a.as_union(
    lambda u: u.tags(lambda t: t.at(0).as_scalar("p")).tags(lambda t: t.at(1).as_scalar("q"))))))  # read back, then written again
assert root_of(Plain.ToPlain(store).OfObject(Choice, same_branch)) == {"pick": {"tags": ["p", "q"]}}
given = store.Choice(lambda c: c.pick(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.tags(container(card.tags))))))
assert root_of(Plain.ToPlain(store).OfObject(Choice, given)) == {"pick": {"tags": ["a", None, "c"]}}  # a container given
with raises(TypeError, match="branch 'tags' takes a container of its schema or a Spec, got int"):
    store.Choice(lambda c: c.pick(lambda p: p.as_value(lambda a: a.as_union(lambda u: u.tags(1)))))
with raises(ReadOnlyError, match="the container is read-only; write through a builder"):
    container(card.tags)(lambda t: t)
# An entry's property may hold a container: entries equal by their containers are one, in a set.
Sets = store.Schemas.OfRelation(lambda r: (
    r
    .name("Sets")
    .links([
        lambda t: t.name("a"),
        lambda t: t.name("b"),
    ])
    .properties([
        lambda t: t.name("tags").schema([Tags]),
        lambda t: t.name("grid").schema([Grid]),
    ])
    .uniques([[]])
))
Marks = store.Schemas.OfObject(lambda o: (
    o
    .name("Marks")
    .ref(True)
    .adjacencies([
        lambda t: t.name("sets").relation([Sets]).me("a"),
        lambda t: t.name("set_by").relation([Sets]).me("b"),
    ])
))
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot())  # store's schemas, as they are now: other's own
two = lambda t: t.at(0).as_scalar("a") and t.at(1).as_scalar("b")
cell = lambda g: g.at((0, 1)).as_scalar(1.5)
target = store.Marks(lambda m: m)
marks = store.Marks(lambda m: m.sets(lambda adj: (
    adj
    .insert(lambda e: e.b(target).tags(lambda p: p.as_value(lambda a: a.as_container(two))).grid(lambda p: p.as_value(lambda a: a.as_container(cell))))
    .insert(lambda e: e.b(target).tags(lambda p: p.as_value(lambda a: a.as_container(two))).grid(lambda p: p.as_value(lambda a: a.as_container(cell))))
)))
row = {"b": ref(1), "tags": ["a", "b"], "grid": [[None, 1.5]]}
assert len(marks.sets) == 1 and root_of(Plain.ToPlain(store).OfObject(Marks, marks)) == {"sets": [row]}
doubled = {"root": {"n": 0}, "objects": [listed("Marks", 0, {"sets": [row, row]}), listed("Marks", 1)]}
assert len(Plain.FromPlain(other).OfObject(other.Marks, doubled).sets) == 1  # read once, by value
assert len(container(note.s)) == 2  # a dense container by text: the elements present
emptied = Plain.FromPlain(other).OfObject(other.Note, single("Note", {"b": [{"index": "AQ=="}], "s": []}))
assert root_of(Plain.ToPlain(other).OfObject(other.Note, emptied)) == {"b": [], "s": []}  # an element absent; empty containers
branch = []
choice.pick.as_value(lambda a: a.as_union(lambda u: branch.append(u.tags) or u))
store.Choice(lambda c: c).pick.as_value(lambda a: a.as_union(lambda u: branch.append(u.tags) or u))
assert len(branch[0]) == 3 and branch[1].as_schema() is None  # a union value not set: its branch reads as empty

## CNT-05 · Malformed containers in snapshots are refused, with a path

In [ ]:
def refused(name, value, message):
    with raises(DecodeError, match=message):
        Plain.FromPlain(store).OfObject(Card, single("Card", {name: value}))


at = "$.objects[0].properties"
refused("tags", {"a": 1}, f"{at}.tags: a dense container must be a list, got dict")
refused("tags", [1], f"{at}.tags[0]: expected str, got int")
refused("grid", [[1.0], 2.0], f"{at}.grid[1]: a container of 2 indices nests 2 lists")
refused("grid", {"shape": [2], "items": [1.0]}, f"{at}.grid: a flat container's shape is 2 sizes, and its items as many as they make")
refused("grid", {"shape": [1, 2], "items": [1.0]}, f"{at}.grid: a flat container's shape is 2 sizes")
refused("grid", {"shape": "2, 3", "items": []}, f"{at}.grid: a flat container's shape is 2 sizes")
refused("grid", {"shape": [1, 1], "items": [True]}, f"{at}.grid.items[0]: expected float, got bool")
refused("sparse", {"at": 1}, f"{at}.sparse: a sparse container must be a list of blocks, got dict")
refused("sparse", [{"at": 1}], f"{at}.sparse[0]: a block is {{'at': index, 'items': [element, ...]}}")
refused("sparse", [{"at": "1", "items": []}], f"{at}.sparse[0].at: expected int, got str")
refused("phones", [], f"{at}.phones: a container keyed by text must be a mapping, got list")
refused("phones", {"home": 1}, f"{at}.phones.home: a value object must be a mapping, got int")
refused("phones", {"home office": 1}, f'{at}.phones["home office"]: a value object must be a mapping, got int')  # not a name: quoted
with raises(DecodeError, match='$.objects[0].properties.m: a container must be a list, got dict'):
    Plain.FromPlain(store).OfObject(Note, single("Note", {"m": {}}))
with raises(DecodeError, match="""$.objects[0].properties.m[0]: an entry of a container is {'index': index, 'element': element}"""):
    Plain.FromPlain(store).OfObject(Note, single("Note", {"m": [{"element": 1}]}))
with raises(DecodeError, match="""$.objects[0].properties.k[0].at: an index has every property of its value object; ['col'] are not set"""):
    Plain.FromPlain(store).OfObject(Note, single("Note", {"k": [{"at": {"row": 1}, "items": [1]}]}))
Deep = store.Schemas.OfObject(lambda o: o.name("Deep").properties([lambda t: t.name("cell").schema([Cell])]))
Odd = store.Schemas.OfContainer(lambda c: c.name("Odd").index([Deep]).element([Int]))  # entries: its index is neither ordered nor hashable
Rare = store.Schemas.OfObject(lambda o: o.name("Rare").ref(True).properties([lambda t: t.name("odd").schema([Odd])]))
with raises(DecodeError, match="""$.objects[0].properties.odd[0].index: a container whose index is neither ordered nor hashable is not read from snapshots yet"""):
    Plain.FromPlain(store).OfObject(Rare, single("Rare", {"odd": [{"index": {}, "element": 1}]}))

## CNT-06 · An index is checked against the index's schema: a scalar of its host type, or a tuple of a value object index's properties; a dense container's is not negative

In [ ]:
def at(name, index):
    store.Card(lambda c: getattr(c, name)(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(index)))))


with raises(TypeError, match="property 'tags': an index is a int, got str"):
    at("tags", "0")
with raises(TypeError, match="property 'tags': an index is a int, got bool"):
    at("tags", True)
with raises(TypeError, match="property 'grid': an index is a tuple of int, int, got [1]"):
    at("grid", (1,))
with raises(TypeError, match="property 'grid': an index is a tuple of int, int, got [1, '2']"):
    at("grid", (1, "2"))
with raises(ValueError, match="property 'tags': a dense container's index is not negative, got -1"):
    at("tags", -1)
with raises(ValueError, match="property 'grid': a dense container's index is not negative, got [0, -1]"):
    at("grid", (0, -1))
store.builder("Card").sparse(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(-5).as_scalar("ok"))))  # sparse: any int
deep = []
store.builder("Rare").odd(lambda p: p.as_value(lambda a: a.as_container(lambda c: deep.append(c))))
with raises(TypeError, match="property 'odd': an index is a value of its index type, got int"):
    deep[0].at(1)
# An index of a value object that is neither ordered nor hashable: given as a value object, written as one.
Keyed = store.Schemas.OfObject(lambda o: (
    o
    .name("Keyed")
    .ref(True)
    .properties([
        lambda t: t.name("key").schema([Deep]),
        lambda t: t.name("odd").schema([Odd]),
    ])
))
keyed = store.Keyed(lambda k: k.key(lambda p: p.as_object(lambda k: k.cell(lambda q: q.as_object(lambda c: c.row(1).col(2))))))
key = []
keyed.key.as_object(lambda k: key.append(k) or k)
keyed = store.Keyed(keyed, lambda k: k.odd(lambda p: p.as_value(lambda a: a.as_container(lambda c: c.at(key[0]).as_scalar(7)))))
assert root_of(Plain.ToPlain(store).OfObject(Keyed, keyed))["odd"] == [{"index": {"cell": {"row": 1, "col": 2}}, "element": 7}]

## CNT-07 · Validation and comparison: each element checked at its index; containers compared by index and element, by value in uniqueness; different index or element schemas incomparable

In [ ]:
into = lambda cb: lambda p: p.as_value(lambda a: a.as_container(cb))
put = lambda pairs, write: into(lambda c: [write(c.at(i), v) for i, v in pairs])
Line = store.Schemas.OfObject(lambda o: o.name("Line").properties([lambda t: t.name("number").schema([Text])]))
Words = store.Schemas.OfContainer(lambda c: c.name("Words").index([Int]).element([Text]))
Book = store.Schemas.OfContainer(lambda c: c.index([Text]).element([Line]))
Table = store.Schemas.OfContainer(lambda c: c.name("Table").index([Cell]).element([Float]))
Nest = store.Schemas.OfContainer(lambda c: c.index([Int]).element([Words]))
ByDeep = store.Schemas.OfContainer(lambda c: c.name("ByDeep").index([Deep]).element([Int]))
Labels = store.Schemas.OfRelation(lambda r: (
    r
    .name("Labels")
    .links([
        lambda t: t.name("shelf"),
        lambda t: t.name("label"),
    ])
    .properties([
        lambda t: t.name("words").schema([Words]),
        lambda t: t.name("deep").schema([ByDeep]),
        lambda t: t.name("table").schema([Table]),
        lambda t: t.name("line").schema([Line]),
    ])
    .uniques("label")
))
Shelf = store.Schemas.OfObject(lambda o: (
    o
    .name("Shelf")
    .ref(True)
    .properties([
        lambda t: t.name("words").schema([Words]),
        lambda t: t.name("book").schema([Book]),
        lambda t: t.name("table").schema([Table]),
        lambda t: t.name("nest").schema([Nest]),
        lambda t: t.name("deep").schema([ByDeep]),
        lambda t: t.name("key").schema([Deep]),
    ])
    .adjacencies([
        lambda t: t.name("labels").relation([Labels]).me("shelf"),
        lambda t: t.name("labelled").relation([Labels]).me("label"),
    ])
))


def shelf(words, book, row):
    made = store.Shelf(lambda s: (
        s
        .words(put(words, lambda e, v: e.as_scalar(v)))
        .book(put(book, lambda e, v: e.as_object(lambda r: r.number(v))))
        .table(put([((0, 1), 0.5)], lambda e, v: e.as_scalar(v)))
        .nest(put([(0, "x")], lambda e, v: e.as_container(lambda w: w.at(1).as_scalar(v))))
        .key(lambda p: p.as_object(lambda k: k.cell(lambda q: q.as_object(lambda c: c.row(row).col(0)))))
    ))
    return store.Shelf(made, lambda s: s.deep(put([(held(made.key), 1)], lambda e, v: e.as_scalar(v))))


def held(read):
    found = []
    read.as_object(lambda k: found.append(k) or k)
    return found[0]


def compare(x, y):
    return Comparison.OfObject(Shelf, x, value=True).compare(Comparison.OfObject(Shelf, y, value=True))


a = shelf([(0, "a"), (2, "c")], [("home", "1"), ("work", "2")], 1)
assert compare(a, shelf([(0, "a"), (2, "c")], [("work", "2"), ("home", "1")], 1)) == 0  # a dict's order is not compared
assert compare(a, shelf([(0, "a"), (1, "c")], [("home", "1"), ("work", "2")], 1)) is None  # an element at another index
assert compare(a, shelf([(0, "a"), (2, "x")], [("home", "1"), ("work", "2")], 1)) is None  # another element
assert compare(a, shelf([(0, "a")], [("home", "1"), ("work", "2")], 1)) is None  # fewer elements
assert compare(a, shelf([(0, "a"), (2, "c")], [("home", "1"), ("work", "2")], 2)) is None  # another value object index
words = container(a.words)
Other = store.Schemas.OfContainer(lambda c: c.index([Text]).element([Text]))  # another index type
Counts = store.Schemas.OfContainer(lambda c: c.index([Int]).element([Int]))  # another element type
assert Comparison.OfContainer(Words, words).compare(Comparison.OfContainer(Words, container(a.words))) == 0
assert Comparison.OfContainer(Words).compare(Comparison.OfContainer(Words)) == 0  # absent, both
assert Comparison.OfContainer(Words, words).compare(Comparison.OfContainer(Words)) is None
assert Comparison.OfContainer(Words, words).compare(Comparison.OfContainer(Other)) is None
assert Comparison.OfContainer(Words, words).compare(Comparison.OfContainer(Counts)) is None
# Uniqueness compares containers by value: two entries of one shelf with equal containers name one label.
b, c = shelf([], [], 1), shelf([], [], 1)
labelled = lambda target, word: lambda x: x.label(target).words(put([(0, word)], lambda e, v: e.as_scalar(v))).deep(
    put([(held(a.key), 2)], lambda e, v: e.as_scalar(v))).table(put([((0, 1), 0.5)], lambda e, v: e.as_scalar(v)))
store.Shelf(a, lambda s: s.labels(lambda r: r.insert(labelled(b, "w")).insert(labelled(c, "w")).insert(labelled(c, "v"))))
unique = "unique(label) violated: entries agreeing on ['deep', 'line', 'shelf', 'table', 'words'] differ on ['label']"  # no line set
assert Validators.Validate(store).OfObject(Shelf, a) == [unique]
# Validation checks each element, at its index.
store.Schemas.OfContainer(Words, lambda c: c.element([Int]))
store.Schemas.OfContainer(Table, lambda c: c.element([Int]))
store.Schemas.OfObject(Line, lambda o: o.properties([lambda t: t.name("number").schema([Int])]))
assert Validators.Validate(store).OfObject(Shelf, a) == [
    "Shelf#0.words[0]: expected int, got str", "Shelf#0.words[2]: expected int, got str",
    "Shelf#0.book['home'].number: expected int, got str", "Shelf#0.book['work'].number: expected int, got str",
    "Shelf#0.table[[0, 1]]: expected int, got float", "Shelf#0.nest[0][1]: expected int, got str",
    "Shelf#0.labels[0].words[0]: expected int, got str", "Shelf#0.labels[0].table[[0, 1]]: expected int, got float",
    "Shelf#0.labels[1].words[0]: expected int, got str", "Shelf#0.labels[1].table[[0, 1]]: expected int, got float",
    "Shelf#0.labels[2].words[0]: expected int, got str", "Shelf#0.labels[2].table[[0, 1]]: expected int, got float",
    unique]